# Phase 4 笔记本 1：非均相催化中的量子计算

> **阅读**：acs.chemrev.8b00803.pdf（Cao 等，2019）

## 问题：DFT 在强关联过渡金属活性位上失效

- **过渡金属氧化物**（FeO、NiO）：U/t ~ 8–12，DFT 误差约 30–50%
- **单原子催化剂**（Fe-N4、Co-N4）：自旋交叉、自由基中间体
- DFT+U 具有经验性——U 参数随体系而变且缺乏系统控制

**量子思路**：对 TM 活性位用 VQE/SQD，体相/环境用经典 DFT 嵌入

```
完整催化剂模型：
  [经典 DFT：体相载体、远场环境]
       +--> [VQE：Fe-N4 活性位，10–20 量子比特] <--+
  [经典 DFT：活性区外的吸附物片段]
```


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 双格点 Hubbard 模型 — 过渡金属氧化物催化的最小模型
# 参数 U/t = 在位排斥与带宽之比
# U/t 增大：金属性 → Mott 绝缘（类似 FeO、NiO）
t = 1.0
Uot = np.linspace(0, 15, 300)
# 精确单重态基态能量
E_exact = 0.5*Uot*t - np.sqrt((0.5*Uot*t)**2 + 4*t**2)
# Hartree-Fock（类 DFT）：E_HF = -2t + U/2
E_hf = -2*t + 0.5*Uot*t
# 关联误差（百分比）
corr_pct = np.abs((E_exact - E_hf) / np.abs(E_exact)) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(Uot, E_exact, 'r-', lw=2.5, label='精确（FCI / VQE 目标）')
ax1.plot(Uot, E_hf, 'b--', lw=2, label='HF（无关联的类 DFT）')
ax1.fill_betweenx([-12, 2], 7, 15, alpha=0.15, color='red', label='Fe-N4 区间（U/t ~ 8–10）')
ax1.set_xlabel('U/t'); ax1.set_ylabel('能量（t 单位）')
ax1.set_title('Hubbard 模型：HF 与精确解')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(Uot, corr_pct, 'r-', lw=2.5)
ax2.axhline(5, color='orange', ls='--', lw=1.5, label='5% 误差阈值')
ax2.fill_betweenx([0, 100], 7, 15, alpha=0.15, color='red', label='TM 氧化物 / SAC 区间')
ax2.set_xlabel('U/t'); ax2.set_ylabel('HF 误差 (%)')
ax2.set_title('过渡金属体系中 DFT 的关联误差')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('hubbard_dft_failure.png', dpi=150, bbox_inches='tight')
plt.show()

# Representative U/t values
tm_systems = [
    ('FeO（氧化铁）',           8.0, 'SAC 载体'),
    ('Fe-N4 单原子',           10.0, '主要目标体系'),
    ('Co-N4 单原子',            8.5, 'CO2RR 催化剂'),
    ('CeO2 氧空位',             5.5, 'CO 氧化'),
    ('Cu 金属',                 1.5, 'DFT 表现良好'),
]
print('过渡金属体系：U/t 与 DFT 误差')
for name, ut, note in tm_systems:
    err = corr_pct[np.argmin(np.abs(Uot - ut))]
    mark = ' <-- 量子计算目标' if ut > 6 else ''
    print(f'  {name:<28} U/t={ut:.1f}  HF_err~{err:.0f}%  {note}{mark}')


## 2. Fe-N4 SAC 活性位：自旋态

Fe-N4 存在多个近简并自旋态（S=0,1,2）。
DFT 若选错自旋态 → O2 吸附能错误 → ORR 活性预测错误。
VQE 可给出精确基态及完整自旋谱。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product as iproduct
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from scipy.optimize import minimize

# Fe 模型：2 个 d 轨道（dz2、dx2-y2），CAS(2,2) → 4 量子比特
delta, J, U = 0.3, 0.15, 0.8   # 晶体场、交换、在位 U（Ha）
I=np.eye(2); X=np.array([[0,1],[1,0]]); Y=np.array([[0,-1j],[1j,0]]); Z=np.array([[1,0],[0,-1]])
def k4(a,b,c,d): return np.kron(np.kron(np.kron(a,b),c),d)

n0u=k4((I-Z)/2,I,I,I); n0d=k4(I,(I-Z)/2,I,I)
n1u=k4(I,I,(I-Z)/2,I); n1d=k4(I,I,I,(I-Z)/2)
Sz_op = 0.5*(k4(Z,I,I,I)+k4(I,Z,I,I)+k4(I,I,Z,I)+k4(I,I,I,Z))

H_Fe  = (delta/2)*(n0u+n0d-n1u-n1d) + U*(n0u@n0d + n1u@n1d)
H_Fe -= J*0.5*(k4(X,I,X,I)+k4(Y,I,Y,I)+k4(I,X,I,X)+k4(I,Y,I,Y))
H_Fe  = np.real(H_Fe)

evals, evecs = np.linalg.eigh(H_Fe)

print('Fe-N4 模型：能谱')
print(f'{"态":<6} {"E(Ha)":<12} {"dE(eV)":<10} {"Sz":<8} 指认')
print('-'*50)
for i in range(6):
    psi = evecs[:,i]
    Sz_i = float(np.real(psi.conj() @ Sz_op @ psi))
    dE = (evals[i]-evals[0])*27.21
    spin_label = '单重态' if abs(Sz_i) < 0.1 else ('三重态' if abs(Sz_i) < 1.1 else '五重态')
    print(f'{i:<6} {evals[i]:<12.4f} {dE:<10.3f} {Sz_i:<8.2f} {spin_label}')

print()
print('DFT 问题：自旋态错误 → O2 吸附能偏差约 0.3–0.8 eV')

# 构造 Pauli 哈密顿量
pd = {'I':I,'X':X,'Y':Y,'Z':Z}
terms = []
for lbl in iproduct('IXYZ', repeat=4):
    P = pd[lbl[0]]
    for l in lbl[1:]: P = np.kron(P, pd[l])
    c = np.real(np.trace(P.conj().T @ H_Fe)) / 16
    if abs(c) > 1e-10: terms.append((''.join(lbl), c))
H_pauli = SparsePauliOp.from_list(terms)
print(f'Pauli 项数: {len(H_pauli)}')

# 硬件高效拟设 VQE（4 量子比特，2 层）
pv = ParameterVector('t', 12)
hea = QuantumCircuit(4)
hea.x([0,1])  # HF 初态：2 个电子在轨道 0
for i in range(4): hea.ry(pv[i], i)
hea.cx(0,1); hea.cx(1,2); hea.cx(2,3)
for i in range(4): hea.ry(pv[i+4], i)
hea.cx(0,2); hea.cx(1,3)
for i in range(4): hea.ry(pv[i+8], i)

est = StatevectorEstimator()
hist = []
def cost(th):
    E = float(est.run([(hea,H_pauli,[th])]).result()[0].data.evs[0])
    hist.append(E); return E

np.random.seed(42)
res = minimize(cost, np.random.uniform(-0.3,0.3,12), method='COBYLA',
               options={'maxiter':600,'rhobeg':0.3})

plt.figure(figsize=(9,4))
plt.plot(hist,'b-',lw=1.5,alpha=0.8,label='VQE 优化')
plt.axhline(evals[0],color='r',ls='--',lw=2,label=f'FCI: {evals[0]:.4f} Ha')
plt.xlabel('迭代次数'); plt.ylabel('能量 (Ha)')
plt.title('Fe-N4 活性位模型的 VQE（4 量子比特，HEA）')
plt.legend(); plt.grid(True,alpha=0.3); plt.tight_layout()
plt.savefig('FeN4_VQE.png',dpi=150,bbox_inches='tight')
plt.show()

err = abs(res.fun - evals[0])*1000
print(f'VQE: {res.fun:.6f} Ha  |  FCI: {evals[0]:.6f} Ha  |  误差: {err:.2f} mHa')
print('化学精度（< 1.6 mHa）:' + (' 是' if err < 1.6 else ' 否 — 可改用 UCCSD 拟设'))


## 3. 完整研究流程（模板）

```python
# 步骤 1：PySCF DFT 优化 + CASSCF 活性空间选取
from pyscf import gto, scf, mcscf
mol = gto.M(atom=fe_n4_xyz, basis='def2-svp', spin=4, charge=0)
mf = scf.ROHF(mol).run()
mc = mcscf.CASSCF(mf, 9, 9)   # CAS(9e, 9orbs)：Fe 3d + N 孤对
mc.kernel()
# NOON 判断哪些轨道强关联
noon = mc.mo_occ[mc.ncore : mc.ncore+9]

# 步骤 2：Qiskit Nature
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit_nature.second_q.mappers import ParityMapper
driver = PySCFDriver(atom=fe_n4_xyz, basis='def2-svp', spin=4)
problem = ActiveSpaceTransformer(9, 9).transform(driver.run())
mapper  = ParityMapper(problem.num_particles)   # 奇偶映射可省 2 个量子比特
H_qubit = mapper.map(problem.second_q_ops()[0])
print(f'量子比特数: {H_qubit.num_qubits}')  # 奇偶约化后约 16

# 步骤 3：VQE 或 SQD
# 选项 A：UCCSD（更精确）
# 选项 B：SQD（更浅，更适合真实硬件）

# 步骤 4：计算吸附能
E_Fe = vqe(Fe_N4_clean)
E_FeO2 = vqe(Fe_N4_with_O2)
E_O2_gas = dft_energy(O2)  # O2 分子用经典 DFT 即可
DeltaE_ads = E_FeO2 - E_Fe - E_O2_gas
```

## 4. 研究路线图

| 年份 | 目标 | 目标期刊 |
|------|------|----------|
| 1 | Fe/Co-N4 模型的 VQE 基准（2–4 量子比特） | J. Chem. Theory Comput. |
| 2 | DMET：DFT（slab）+ VQE（活性位），ORR 研究 | JACS, ACS Catalysis |
| 3+ | 真实 IBM 硬件、SQD、更大活性空间 | Nature Chemistry |

**下一篇**：笔记本 2 — 量子计算辅助力场开发（与 ReaxFF 工作的直接延伸）
